# Gate Defence + Tobii Eye Tracker 5: merge and visualise

Inputs:
- the game's data as a **ZIP** (the session files: `..._events.csv`, `..._trials.csv`, `..._session.json`, ...)
- the Tobii recording, **`gaze_data_....txt`**, from `enhanced-tobii-eyetracker\data`

Run the cells top to bottom (**Run → Run All Cells**). Results are saved to the output folder set below.

How the merge works: both programs ran on the same PC. The game logged every event with the PC clock (`unix_time`), and eyeTrack.exe logged every gaze sample with the PC clock (`human_time`) and the tracker's clock (`tobii_timestamp_us`). Lining up the clocks puts every gaze sample on the game's timeline.

## 0. Check and install the libraries
Anything missing is installed into the Python this notebook is using (a few minutes the first time on a new computer), then the notebook carries on.

In [ ]:
import importlib
import importlib.metadata as md
import subprocess
import sys

# import name -> (pip name, minimum version)
REQUIRED = {
    "numpy": ("numpy", None),
    "pandas": ("pandas", None),
    "matplotlib": ("matplotlib", None),
    "polars": ("polars", None),
    "pymovements": ("pymovements", "0.28"),
}
# In mainland China the default package server can be slow; set a mirror here if needed, e.g.
# PIP_INDEX = "https://pypi.tuna.tsinghua.edu.cn/simple"
PIP_INDEX = None


def version_ok(installed, minimum):
    if minimum is None:
        return True
    to_tuple = lambda v: tuple(int(p) for p in v.split(".")[:2] if p.isdigit())
    return to_tuple(installed) >= to_tuple(minimum)


missing = []
for module, (package, minimum) in REQUIRED.items():
    try:
        installed = md.version(package)
        if not version_ok(installed, minimum):
            print(f"{package} {installed} is too old (need {minimum}+)")
            missing.append(f"{package}>={minimum}")
        else:
            print(f"ok       {package} {installed}")
    except md.PackageNotFoundError:
        print(f"missing  {package}")
        missing.append(package if minimum is None else f"{package}>={minimum}")

if missing:
    cmd = [sys.executable, "-m", "pip", "install", "--upgrade", *missing]
    if PIP_INDEX:
        cmd += ["-i", PIP_INDEX]
    print()
    print("Installing:", " ".join(missing))
    print("using", sys.executable)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-2000:], result.stderr[-2000:])
        raise RuntimeError("pip could not install the libraries: see the message above. "
                           "If the download is slow or blocked, set PIP_INDEX to a mirror and run this cell again.")
    importlib.invalidate_caches()
    print("Installed.")
else:
    print()
    print("All libraries are installed.")

## 1. Settings

In [ ]:
from pathlib import Path

GAME_ZIP = Path("data.zip")                          # the game's data ZIP
GAZE_TXT = Path("gaze_data_20261020_163440.txt")     # the Tobii recording
OUT = Path("output")                                 # where results are saved

SESSION = None          # None = newest session in the ZIP, or part of its name, e.g. "P003"

# Screen and seating (Lenovo Y9000P, 16 inch)
SCREEN_PX = (2560, 1600)   # screen resolution
SCREEN_CM = (34.5, 21.5)   # visible screen width and height in cm
DISTANCE_CM = 50           # eyes to screen in cm

CENTRE_RADIUS = 200        # gaze nearer than this to the player (game pixels) counts as "centre"
WINDOW_MS = 1500           # analysis window after each bang
DWELL_MS = 600             # dwell window after each bang (robot appears at 300 ms)
LEVEL, WAVE = 3, 1         # wave shown in the detailed plots (one wave = 6 robots)

## 2. Libraries

In [ ]:
import json
import warnings
import zipfile

warnings.filterwarnings("ignore")
try:
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import polars as pl
    import pymovements as pm
except ImportError as e:
    raise ImportError(f"{e}. Choose Kernel -> Restart Kernel and Run All Cells "
                      "(needed if a library was updated while it was already loaded).") from None
from IPython.display import display

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.dpi"] = 110
OUT.mkdir(exist_ok=True)
REF_W, REF_H = 1920, 1080          # the game's own coordinate frame
GATE_NAMES = {0: "left", 1: "ahead", 2: "right", -1: "bottom", -2: "centre"}

## 3. Unpack the game ZIP and load the session

In [ ]:
unzipped = OUT / "game_data"
with zipfile.ZipFile(GAME_ZIP) as z:
    for name in z.namelist():
        if name.endswith((".csv", ".json")):        # skip Godot's .translation / .import files
            target = unzipped / Path(name).name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(z.read(name))

event_files = sorted(unzipped.glob("*_events.csv"))
if SESSION:
    event_files = [f for f in event_files if SESSION in f.name]
if not event_files:
    raise FileNotFoundError("No *_events.csv in the ZIP" + (f" matching {SESSION!r}" if SESSION else ""))
base = Path(str(event_files[-1])[: -len("_events.csv")])
NAME = base.name
print("Sessions in ZIP:", [f.name[:-len("_events.csv")] for f in sorted(unzipped.glob("*_events.csv"))])
print("Using:", NAME)

events = pd.read_csv(f"{base}_events.csv")
trials = pd.read_csv(f"{base}_trials.csv")
session = json.loads(Path(f"{base}_session.json").read_text())
arena = session["config"]["arena"]
cx, cy = session["centre_px"]
print(f"{len(trials)} robots, {len(events)} events, participant {trials['pid'].iloc[0]}, "
      f"reliability condition {trials['reliability_condition'].iloc[0]}")

## 4. Load the gaze recording

In [ ]:
gaze = pd.read_csv(GAZE_TXT, sep="\t")

# human_time is local time; convert it to unix microseconds like the game's unix_time
naive = pd.to_datetime(gaze["human_time"], format="%Y-%m-%d %H:%M:%S.%f")
as_utc_us = (naive - pd.Timestamp("1970-01-01")) // pd.Timedelta("1us")
utc_offset_us = round(naive.iloc[0].to_pydatetime().timestamp() * 1e6) - int(as_utc_us.iloc[0])
gaze["wall_us"] = as_utc_us + utc_offset_us

rate = 1e6 / np.median(np.diff(gaze["tobii_timestamp_us"]))
print(f"{len(gaze)} samples, {rate:.1f} Hz, recorded {gaze['human_time'].iloc[0]} to {gaze['human_time'].iloc[-1]}")

## 5. Line up the clocks
- Game: `t_us` (game timer) and `unix_time` (PC clock) on every event, so their difference is fixed.
- Tracker: `tobii_timestamp_us` (when the eye was measured) and `human_time` (when the row was written). eyeTrack.exe writes a burst of buffered samples when recording starts, so the median difference is used, which ignores that burst.

Each sample's time on the game timer = tracker time + (PC clock − tracker clock) − (PC clock − game timer).

In [ ]:
events["unix_us"] = (events["unix_time"] * 1e6).round()
game_offset = np.median(events["unix_us"] - events["t_us"])                       # PC clock - game timer
tracker_offset = np.median(gaze["wall_us"] - gaze["tobii_timestamp_us"])          # PC clock - tracker clock
gaze["t_us"] = gaze["tobii_timestamp_us"] + tracker_offset - game_offset
gaze = gaze.sort_values("t_us").reset_index(drop=True)

# checks
start_us, end_us = events["t_us"].min(), events["t_us"].max()
inside = gaze[(gaze["t_us"] >= start_us) & (gaze["t_us"] <= end_us)]
if len(inside) < 0.5 * (end_us - start_us) / 1e6 * rate:
    raise ValueError("This gaze file does not cover the session: check that the files belong together.")
steady = gaze["wall_us"] - gaze["tobii_timestamp_us"] - tracker_offset
print(f"Write delay after the start burst: {np.median(np.abs(steady)) / 1000:.1f} ms (median)")
print(f"Gaze in the session: {len(inside)} samples, data loss "
      f"{100 * (1 - len(inside) / ((end_us - start_us) / 1e6 * rate)):.1f}%")

## 6. Gaze position in game pixels, and gate areas
The game draws its 1920×1080 frame scaled to fit the screen, with black bars on a 16:10 screen. The tracker gives positions from 0 to 1 across the whole screen.

Gate areas are the left, top and right quarters around the player, beyond `CENTRE_RADIUS`. Codes as in the game: 0 left, 1 ahead, 2 right, −1 bottom, −2 centre.

In [ ]:
sw, sh = SCREEN_PX
scale = min(sw / REF_W, sh / REF_H)
off_x, off_y = (sw - REF_W * scale) / 2, (sh - REF_H * scale) / 2
gaze["x"] = (gaze["x_position"] * sw - off_x) / scale
gaze["y"] = (gaze["y_position"] * sh - off_y) / scale

dx, dy = gaze["x"] - cx, gaze["y"] - cy
angle = (np.degrees(np.arctan2(dx, -dy)) + 360) % 360          # clockwise from straight up
sector = np.select([(angle >= 315) | (angle < 45), (angle >= 45) & (angle < 135),
                    (angle >= 225) & (angle < 315)], [1, 2, 0], default=-1)
gaze["gate"] = np.where(np.hypot(dx, dy) < CENTRE_RADIUS, -2, sector)

# degrees of visual angle between two points in game pixels
cm_per_px = SCREEN_CM[0] / sw * scale
def eye_vectors(x, y):
    sx = (np.asarray(x, float) - REF_W / 2) * cm_per_px
    sy = (np.asarray(y, float) - REF_H / 2) * cm_per_px
    v = np.stack([sx, sy, np.full_like(sx, DISTANCE_CM)], -1)
    return v / np.linalg.norm(v, axis=-1, keepdims=True)
def angle_deg(x1, y1, x2, y2):
    return np.degrees(np.arccos(np.clip(np.sum(eye_vectors(x1, y1) * eye_vectors(x2, y2), -1), -1, 1)))

t = gaze["t_us"].to_numpy()
gx, gy, gg = gaze["x"].to_numpy(), gaze["y"].to_numpy(), gaze["gate"].to_numpy()
def window(a_us, b_us):
    i, j = np.searchsorted(t, [a_us, b_us])
    return slice(i, j)
def gate_at(t_us, max_age_ms=100):
    i = np.searchsorted(t, t_us, side="right") - 1
    return gg[i] if i >= 0 and t_us - t[i] <= max_age_ms * 1000 else np.nan

gaze[["t_us", "x", "y", "gate"]].round(1).to_csv(OUT / f"{NAME}_gaze.csv", index=False)
gaze["gate"].map(GATE_NAMES).value_counts(normalize=True).round(3)

## 7. Data quality: the game's 9-dot validation
Accuracy = how far the average gaze was from each dot, in degrees. The dots on the gates' row and column (`at_gate`) matter most, because that is where the game's areas of interest are. `arrival_ms` = how long the eyes took to reach the dot.

In [ ]:
point_ms = float(session["config"]["validation"]["point_ms"])
rows = []
for _, e in events[events["event"] == "validation_point"].iterrows():
    d = json.loads(e["details"])
    w = window(e["t_us"] + 400_000, e["t_us"] + (point_ms - 50) * 1000)
    row = {"tag": d["tag"], "x": d["x"], "y": d["y"], "n": w.stop - w.start,
           "at_gate": abs(d["y"] - cy) < 1 or (abs(d["x"] - cx) < 1 and d["y"] <= cy)}
    if row["n"]:
        mx, my = gx[w].mean(), gy[w].mean()
        row["accuracy_deg"] = float(angle_deg(mx, my, d["x"], d["y"]))
        s2s = angle_deg(gx[w][:-1], gy[w][:-1], gx[w][1:], gy[w][1:])
        row["precision_rms_deg"] = float(np.sqrt(np.mean(s2s ** 2)))
        w2 = window(e["t_us"], e["t_us"] + point_ms * 1000)
        near = np.nonzero(angle_deg(gx[w2], gy[w2], d["x"], d["y"]) < 3)[0]
        row["arrival_ms"] = (t[w2][near[0]] - e["t_us"]) / 1000 if len(near) else np.nan
        row["mean_x"], row["mean_y"] = mx, my
    rows.append(row)
val = pd.DataFrame(rows)
val.round(2).to_csv(OUT / f"{NAME}_validation.csv", index=False)
print(f"Accuracy: {val['accuracy_deg'].mean():.2f} deg (all dots), "
      f"{val.loc[val['at_gate'], 'accuracy_deg'].mean():.2f} deg at the gate positions, "
      f"precision {val['precision_rms_deg'].mean():.3f} deg")
display(val.round(2))

fig, ax = plt.subplots(figsize=(8, 4.5))
for _, r in val.iterrows():
    ax.plot(r["x"], r["y"], "k+", ms=14, mew=2)
    ax.plot([r["x"], r["mean_x"]], [r["y"], r["mean_y"]], color="C3")
    ax.plot(r["mean_x"], r["mean_y"], "o", color="C3")
    ax.annotate(f"{r['accuracy_deg']:.1f}°", (r["x"], r["y"]), xytext=(6, -12), textcoords="offset points", fontsize=8)
ax.set_xlim(0, REF_W); ax.set_ylim(REF_H, 0); ax.set_aspect("equal")
ax.set_title("Validation: + dot, ● average gaze"); plt.show()

## 8. Gaze measures for every robot
Times are from the bang (for silent robots, from when the bang would have been).
- `gaze_gate_at_cue`: gate area the eyes were in at the bang
- `first_gaze_gate`, `first_gaze_ms`: first gate area entered after the bang (different from the one at the bang)
- `first_gaze_before_emerge`: 1 if that happened before the robot appeared
- `first_gaze_to_sound / _target / _sensor`: first look went to the bang's / robot's / blip's gate
- `gaze_to_target_ms`: when the eyes reached the robot's gate
- `dwell_*_pct`: share of the first 600 ms on each gate
- `n_gate_switches`: changes between gates in the 1.5 s after the bang
- `pre_cue_sensor_pct`: share of time on the blip's gate before the bang
- `clean`: only one robot on screen; `gaze_ok`: enough gaze recorded

In [ ]:
cue_t = trials["t_cue_us"].astype(float).copy()
silent = cue_t.isna()
cue_t[silent] = trials.loc[silent, "t_emerge_us"] - trials.loc[silent, "soa_ms"].fillna(300) * 1000
pct = lambda m: 100 * float(np.mean(m)) if len(m) else np.nan

out = []
for i, r in trials.iterrows():
    tc, te, ts = cue_t[i], float(r["t_emerge_us"]), float(r["t_scene_us"])
    m = {}
    if np.isnan(tc):
        out.append(m); continue
    cue_g = r["cue_gate"] if r["cue_gate"] >= 0 else np.nan
    tgt, sen = r["correct_gate"], r["sensor_gate"]
    w = window(tc, tc + WINDOW_MS * 1000)
    gw, tw = gg[w], t[w]
    at_cue = gate_at(tc)
    m["gaze_coverage"] = min(1.0, len(gw) / (rate * WINDOW_MS / 1000))
    m["gaze_ok"] = int(m["gaze_coverage"] >= 0.5 and not np.isnan(at_cue))
    m["gaze_gate_at_cue"] = at_cue
    if not np.isnan(ts) and tc > ts:
        m["pre_cue_sensor_pct"] = pct(gg[window(ts, tc)] == sen)
    moved = np.nonzero((gw >= 0) & (gw != at_cue))[0]
    if len(moved):
        k = moved[0]
        m["first_gaze_gate"] = gw[k]
        m["first_gaze_ms"] = (tw[k] - tc) / 1000
        m["first_gaze_before_emerge"] = int(tw[k] < te)
        m["first_gaze_to_sound"] = np.nan if np.isnan(cue_g) else int(gw[k] == cue_g)
        m["first_gaze_to_target"] = int(gw[k] == tgt)
        m["first_gaze_to_sensor"] = int(gw[k] == sen)
    on_t = np.nonzero(gw == tgt)[0]
    if at_cue == tgt:
        m["gaze_to_target_ms"] = 0.0
    elif len(on_t):
        m["gaze_to_target_ms"] = (tw[on_t[0]] - tc) / 1000
    gd = gg[window(tc, tc + DWELL_MS * 1000)]
    m["dwell_sound_pct"] = np.nan if np.isnan(cue_g) else pct(gd == cue_g)
    m["dwell_target_pct"] = pct(gd == tgt)
    m["dwell_sensor_pct"] = pct(gd == sen)
    m["dwell_centre_pct"] = pct(gd == -2)
    only = gw[gw >= 0]
    m["n_gate_switches"] = int(np.sum(only[1:] != only[:-1])) if len(only) else 0
    if r["mode"] == "live" and "first_gaze_ms" in m and pd.notna(r["first_aim_ms"]):
        m["gaze_aim_lead_ms"] = r["first_aim_ms"] - m["first_gaze_ms"]
    out.append(m)

tg = pd.concat([trials, pd.DataFrame(out, index=trials.index)], axis=1)
tg["clean"] = ((tg["others_alive_at_cue"] == 0) & (tg["others_alive_at_emerge"] == 0)).astype(int)
tg["cue_t_us"] = cue_t
tg.to_csv(OUT / f"{NAME}_trials_gaze.csv", index=False)

an = tg[(tg["block"] != "practice") & (tg["clean"] == 1) & (tg["gaze_ok"] == 1)]
print(f"{len(tg[tg['block'] != 'practice'])} robots in the levels, {len(an)} clean with good gaze")
display(tg[["epoch", "reliability", "audio_cond", "cue_gate", "correct_gate", "sensor_gate",
            "gaze_gate_at_cue", "first_gaze_gate", "first_gaze_ms", "first_gaze_to_sound",
            "gaze_to_target_ms", "clean", "gaze_ok"]].head(12))

## 9. Summary per level and bang condition

In [ ]:
summary = an.groupby(["epoch", "reliability", "audio_cond"]).agg(
    robots=("trial_global", "size"),
    first_gaze_to_sound_pct=("first_gaze_to_sound", lambda s: 100 * s.mean()),
    first_gaze_to_target_pct=("first_gaze_to_target", lambda s: 100 * s.mean()),
    before_robot_pct=("first_gaze_before_emerge", lambda s: 100 * s.mean()),
    first_gaze_ms=("first_gaze_ms", "median"),
    gaze_to_target_ms=("gaze_to_target_ms", "median"),
    dwell_sound_pct=("dwell_sound_pct", "mean"),
    gate_switches=("n_gate_switches", "mean"),
).round(1)
conflict = an[an["cue_sensor"] == "conflict"].groupby(["epoch", "reliability"]).agg(
    conflict_robots=("trial_global", "size"),
    followed_sound_pct=("first_gaze_to_sound", lambda s: 100 * s.mean()),
    followed_blip_pct=("first_gaze_to_sensor", lambda s: 100 * s.mean())).round(1)
summary.to_csv(OUT / f"{NAME}_summary.csv")
conflict.to_csv(OUT / f"{NAME}_conflict.csv")
display(summary)
print("Conflict robots (bang and blip point to different gates):")
display(conflict)

## 10. Plots per level

In [ ]:
levels = sorted(an["epoch"].unique())
rel = an.groupby("epoch")["reliability"].first()
lab = lambda e: f"Level {e}\n{100 * rel[e]:.0f}% reliable"

# 10a. first look to the bang's gate
s = an[an["audio_cond"].isin(["valid", "invalid"])].pivot_table(
    index="epoch", columns="audio_cond", values="first_gaze_to_sound", aggfunc="mean").reindex(levels)
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, c in enumerate(["valid", "invalid"]):
    if c in s:
        ax.bar(np.arange(len(levels)) + (i - 0.5) * 0.38, 100 * s[c], 0.38, label=f"{c} bang")
ax.axhline(100 / 3, color="0.6", ls=":", lw=1)
ax.set_xticks(range(len(levels)), [lab(e) for e in levels]); ax.set_ylim(0, 105)
ax.set_ylabel("first look to the bang's gate (%)"); ax.legend(frameon=False)
ax.set_title("Do the eyes follow the bang?")
fig.savefig(OUT / f"{NAME}_first_gaze.png", dpi=150, bbox_inches="tight"); plt.show()

# 10b. cost of a wrong bang
v = an[an["audio_cond"].isin(["valid", "invalid"])].pivot_table(
    index="epoch", columns="audio_cond", values="gaze_to_target_ms", aggfunc="median").reindex(levels)
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, c in enumerate(["valid", "invalid"]):
    if c in v:
        ax.bar(np.arange(len(levels)) + (i - 0.5) * 0.38, v[c], 0.38, label=f"{c} bang")
ax.axhline(300, color="0.5", ls=":", lw=1); ax.text(len(levels) - 0.6, 310, "robot appears", fontsize=8)
ax.set_xticks(range(len(levels)), [lab(e) for e in levels])
ax.set_ylabel("eyes reach the robot's gate (ms after bang, median)"); ax.legend(frameon=False)
ax.set_title("Cost of a wrong bang")
fig.savefig(OUT / f"{NAME}_wrong_bang_cost.png", dpi=150, bbox_inches="tight"); plt.show()

### Time course
Share of gaze on the bang's and the robot's gate, in 50 ms steps from 300 ms before to 1500 ms after the bang.

In [ ]:
BIN = 50
tc_rows = []
for _, r in an.iterrows():
    tc = r["cue_t_us"]
    w = window(tc - 300_000, tc + WINDOW_MS * 1000)
    rel_ms = (t[w] - tc) / 1000
    for b, g in zip(np.floor(rel_ms / BIN) * BIN, gg[w]):
        tc_rows.append({"epoch": r["epoch"], "audio_cond": r["audio_cond"], "bin": b,
                        "on_sound": np.nan if r["cue_gate"] < 0 else float(g == r["cue_gate"]),
                        "on_target": float(g == r["correct_gate"])})
tcd = pd.DataFrame(tc_rows)
tcd.to_csv(OUT / f"{NAME}_timecourse.csv", index=False)

fig, axes = plt.subplots(1, len(levels), figsize=(3.8 * len(levels), 3.6), sharey=True, squeeze=False)
for ax, e in zip(axes[0], levels):
    d = tcd[tcd["epoch"] == e]
    for c, col, style, label in [("valid", "on_sound", "-", "valid: bang's gate"),
                                 ("invalid", "on_sound", "--", "invalid: bang's gate"),
                                 ("invalid", "on_target", ":", "invalid: robot's gate"),
                                 ("silent", "on_target", "-.", "silent: robot's gate")]:
        m = d[d["audio_cond"] == c].groupby("bin")[col].mean()
        if len(m):
            ax.plot(m.index + BIN / 2, 100 * m, style, label=label)
    ax.axvline(0, color="0.5", lw=0.8); ax.axvline(300, color="0.5", lw=0.8, ls=":")
    ax.set_title(lab(e).replace("\n", ", "), fontsize=10); ax.set_xlabel("ms from bang")
axes[0][0].set_ylabel("% of gaze"); axes[0][0].set_ylim(0, 105)
axes[0][-1].legend(frameon=False, fontsize=7)
fig.tight_layout(); fig.savefig(OUT / f"{NAME}_timecourse.png", dpi=150, bbox_inches="tight"); plt.show()

## 11. pymovements: fixations and heatmaps over the arena
The arena is drawn from the game's own settings. pymovements' I-DT fixation detection needs evenly spaced samples, so gaze is put on a 20 ms grid; gaps longer than 75 ms (blinks, lost tracking) stay empty.

In [ ]:
from matplotlib.colors import ListedColormap

def draw_arena(ax):
    # arena from the game settings, drawn underneath everything else
    h, wall, gw_ = arena["half_size_px"], arena["wall_px"], arena["gate_width_px"]
    ax.set_facecolor("#0b0e16")
    ax.add_patch(plt.Rectangle((cx - h - wall, cy - h - wall), 2 * (h + wall), 2 * (h + wall), color="#4a5263", zorder=-3))
    ax.add_patch(plt.Rectangle((cx - h, cy - h), 2 * h, 2 * h, color="#1d2230", zorder=-3))
    for gx_, gy_, w_, h_ in [(cx - h - wall, cy - gw_ / 2, wall, gw_), (cx - gw_ / 2, cy - h - wall, gw_, wall),
                             (cx + h, cy - gw_ / 2, wall, gw_)]:
        ax.add_patch(plt.Rectangle((gx_, gy_), w_, h_, color="#e0a526", zorder=-2))
    ax.add_patch(plt.Circle((cx, cy), 22, color="#3d7bd6", zorder=-2))
    ax.set_xlim(0, REF_W); ax.set_ylim(REF_H, 0); ax.set_aspect("equal")

def to_pm_gaze(sel):
    tt, xx, yy = t[sel] / 1000, gx[sel], gy[sel]
    grid = np.arange(np.ceil(tt[0] / 20) * 20, tt[-1], 20)
    x2, y2 = np.interp(grid, tt, xx), np.interp(grid, tt, yy)
    right = np.clip(np.searchsorted(tt, grid), 1, len(tt) - 1)
    gap = (tt[right] - tt[right - 1]) > 75
    x2[gap] = y2[gap] = np.nan
    t_rel = grid.astype(np.int64) - int(round(start_us / 1000))       # whole ms, evenly spaced
    df = pl.DataFrame({"time": t_rel, "x": x2, "y": y2})
    exp = pm.Experiment(screen_width_px=REF_W, screen_height_px=REF_H, screen_width_cm=REF_W * cm_per_px,
                        screen_height_cm=REF_H * cm_per_px, distance_cm=DISTANCE_CM,
                        origin="upper left", sampling_rate=50)
    g = pm.Gaze(df, experiment=exp, time_column="time", time_unit="ms", pixel_columns=["x", "y"])
    g.pix2deg()
    return g

colors = plt.get_cmap("jet")(np.linspace(0, 1, 256))
colors[:, 3] = np.clip(np.linspace(0, 1, 256) * 2.5, 0, 0.8)       # transparent where nobody looked
heat_cmap = ListedColormap(colors)

def robot_mask(rows):
    keep = np.zeros(len(t), bool)
    for c in rows["cue_t_us"].dropna():
        keep[(t >= c) & (t < c + WINDOW_MS * 1000)] = True
    return keep

main = tg[tg["block"] != "practice"]
fig, axes = plt.subplots(2, 2, figsize=(13, 7.8))
for ax, e in zip(axes.ravel(), sorted(main["epoch"].unique())):
    rows = main[main["epoch"] == e]
    draw_arena(ax)
    pm.plotting.heatmap(to_pm_gaze(robot_mask(rows)), position_column="pixel", gridsize=(48, 27),
                        cmap=heat_cmap, show_cbar=True, cbar_label="time looked (s)", ax=ax)
    conds = ", ".join(f"{n} {c}" for c, n in rows["audio_cond"].value_counts().items())
    ax.set_title(f"Level {e}: {100 * rows['reliability'].iloc[0]:.0f}% reliable ({conds})", fontsize=10)
    ax.set_xlim(0, REF_W); ax.set_ylim(REF_H, 0)
fig.suptitle("Where the eyes were during the robots, per level")
fig.tight_layout(); fig.savefig(OUT / f"{NAME}_heatmap_levels.png", dpi=150, bbox_inches="tight"); plt.show()

### Fixations for the whole session (I-DT, 2°, at least 100 ms)

In [ ]:
session_gaze = to_pm_gaze(slice(None))
session_gaze.detect("idt", dispersion_threshold=2.0, minimum_duration=100)
session_gaze.compute_event_properties(("location", {"position_column": "pixel"}))
fix = session_gaze.events.frame.with_columns(
    pl.col("location").list.get(0).alias("x"), pl.col("location").list.get(1).alias("y"),
).drop("location").filter(pl.col("onset") >= 0).to_pandas()

# tag each fixation with its level, and with the robot if it started within 1.5 s of a bang
blocks = events[events["event"] == "block_start"]
b_t = ((blocks["t_us"] - start_us) / 1000).to_numpy()
b_lvl = [0 if '"practice"' in d else int(json.loads(d)["block"].replace("epoch", "")) for d in blocks["details"]]
fix["level"] = [b_lvl[k] if (k := np.searchsorted(b_t, o, side="right") - 1) >= 0 else np.nan for o in fix["onset"]]
cues = main.assign(c=(main["cue_t_us"] - start_us) / 1000).sort_values("c")
idx = np.searchsorted(cues["c"].to_numpy(), fix["onset"].to_numpy(), side="right") - 1
ok = (idx >= 0) & (fix["onset"].to_numpy() - cues["c"].to_numpy()[np.clip(idx, 0, None)] < WINDOW_MS)
for col in ["trial_global", "audio_cond", "reliability"]:
    fix[col] = np.where(ok, cues[col].to_numpy()[np.clip(idx, 0, None)], None)
fix.to_csv(OUT / f"{NAME}_fixations.csv", index=False)
print(f"{len(fix)} fixations, median {fix['duration'].median():.0f} ms")
display(fix.dropna(subset=["trial_global"]).groupby(["level", "audio_cond"])["duration"]
        .agg(fixations="size", median_ms="median").round(0))

### One wave in detail: gaze path, fixation scanpath and time series
Change `LEVEL, WAVE` in the settings to look at another wave.

In [ ]:
wv = main[(main["epoch"] == LEVEL) & (main["wave"] == WAVE)]
if wv.empty:
    raise ValueError(f"No robots in level {LEVEL}, wave {WAVE}")
a_us, b_us = wv["cue_t_us"].min() - 600_000, wv["t_emerge_us"].max() + WINDOW_MS * 1000
sel = (t >= a_us) & (t <= b_us)
wave_gaze = to_pm_gaze(sel)
wave_gaze.detect("idt", dispersion_threshold=2.0, minimum_duration=100)
wave_gaze.compute_event_properties(("location", {"position_column": "pixel"}))
label = f"level {LEVEL}, wave {WAVE} ({len(wv)} robots, {(b_us - a_us) / 1e6:.1f} s)"

fig, ax = plt.subplots(figsize=(10, 5.6)); draw_arena(ax)
secs = (wave_gaze.samples["time"].to_numpy() - (a_us - start_us) / 1000) / 1000
pm.plotting.traceplot(wave_gaze, position_column="pixel", ax=ax, cval=secs, cmap=plt.get_cmap("plasma"),
                      show_cbar=True, cbar_label="seconds into the wave")
ax.set_xlim(0, REF_W); ax.set_ylim(REF_H, 0); ax.set_title(f"Gaze path (colour = time), {label}")
fig.savefig(OUT / f"{NAME}_wave_trace.png", dpi=150, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(10, 5.6)); draw_arena(ax)
pm.plotting.scanpathplot(wave_gaze, position_column="location", ax=ax, color="yellow", alpha=0.6,
                         arrow_color="white", arrow_rad=0.15)
ax.set_xlim(0, REF_W); ax.set_ylim(REF_H, 0)
ax.set_title(f"Fixations (circle size = duration) and their order, {label}")
fig.savefig(OUT / f"{NAME}_wave_scanpath.png", dpi=150, bbox_inches="tight"); plt.show()

fig, axes = pm.plotting.tsplot(wave_gaze, channels="pixel", n_cols=1, show_events=True, figsize=(13, 4.5),
                               title=f"Gaze position over time, {label}")
for ax in np.atleast_1d(axes).ravel():
    for c, e in zip(wv["cue_t_us"], wv["t_emerge_us"]):
        ax.axvline((c - start_us) / 1000, color="C3", lw=1)
        ax.axvline((e - start_us) / 1000, color="C3", lw=1, ls=":")
np.atleast_1d(axes).ravel()[-1].set_xlabel("ms from session start (red: bang solid, robot dotted; blue: fixations)")
fig.savefig(OUT / f"{NAME}_wave_timeseries.png", dpi=150, bbox_inches="tight"); plt.show()

## 12. Saved files

In [ ]:
for f in sorted(OUT.glob(f"{NAME}_*")):
    print(f.name)